# Task 1: Test Script

This notebook contains **domain data, expected results and checks**, not implementations of the reasoners. The checks become runnable when the `engine.py` provides the starter API.

**Scope:** Task 1 only: one rule base, forward chaining, backward chaining, variables and recursion. Task 2 is separate. This is only for testing, the final world and demonstration is in a separate notebook.

In each section: the cell is run before and after the implementation is ready to assert.



## 1. The test world

The test world is in line with the assignment examples and original documentation (to be safe).
The initial facts describe six parent relationships. Read `parent(x, y)` as “x is a parent of y”.

| Parent | Child |
|---|---|
| mary | ann |
| ann | sam |
| sam | leo |
| mary | bob |
| bob | ivy |
| tom | nina |

Mary's family has two branches. The path Mary → Ann → Sam → Leo forces recursive reasoning beyond one application. Tom and Nina form a disconnected family, giving useful negative checks.

**Predict:** Who are Mary's children, grandchildren and descendants? Does Mary have a relationship to Nina under these rules?


In [1]:

from reasoning.domains import TEST_1_FACTS, TEST_1_RULES


for fact in sorted(TEST_1_FACTS):
    print(fact)


('parent', 'ann', 'sam')
('parent', 'bob', 'ivy')
('parent', 'mary', 'ann')
('parent', 'mary', 'bob')
('parent', 'sam', 'leo')
('parent', 'tom', 'nina')


## 2. Establish expected test results
In order to test the engines ability to predict (forward chain) and plan (backward chain) we manually reason what the results should be.

The rules mean:

1. **Grandparent:** two parent links conclude a grandparent relationship.
2. **Direct ancestry:** every parent is an ancestor.
3. **Recursive ancestry:** if x is a parent of y, and y is an ancestor of z, x is an ancestor of z.

These are inferences about relationships; firing them does not change who anyone's parent is.


In [2]:
# Expected results are stated independently of either reasoner's output.
EXPECTED_GRANDPARENTS = {
    ("grandparent", "mary", "sam"),
    ("grandparent", "mary", "ivy"),
    ("grandparent", "ann", "leo"),
}
EXPECTED_ANCESTORS = {
    ("ancestor", "mary", "ann"),
    ("ancestor", "mary", "sam"),
    ("ancestor", "mary", "leo"),
    ("ancestor", "mary", "bob"),
    ("ancestor", "mary", "ivy"),
    ("ancestor", "ann", "sam"),
    ("ancestor", "ann", "leo"),
    ("ancestor", "sam", "leo"),
    ("ancestor", "bob", "ivy"),
    ("ancestor", "tom", "nina"),
}

EXPECTED_CLOSURE = TEST_1_FACTS | EXPECTED_GRANDPARENTS | EXPECTED_ANCESTORS
assert len(EXPECTED_CLOSURE) == 19
print("Expected closure: 6 parent + 3 grandparent + 10 ancestor facts = 19.")


Expected closure: 6 parent + 3 grandparent + 10 ancestor facts = 19.


## 3. Groundwork and connections

The starter functions to test:

- `forward_chain(wm, rules)` adds conclusions to WM and returns WM.
- `backward_chain(wm, rules, query, subst=None)` yields bindings for proofs and does not change WM.

The starter functions raise `NotImplementedError` until they are implemented. An import failure here means the project modules are not yet in place or on the import path; it is not a reasoning failure.

The display helpers below only format and compare answers. They do not derive facts or prove goals. Bindings are resolved using the supplied `substitute` function because the raw answer may contain variable-to-variable chains.


In [3]:
from reasoning.engine import WorkingMemory, forward_chain, backward_chain, substitute, backward_answers

In [4]:
## Some helper functions for the below:

def show_facts(facts):
    for fact in sorted(facts):
        print(fact)

def explain_check(label, actual, expected):
    if actual != expected:
        print("Missing:", sorted(expected - actual))
        print("Unexpected:", sorted(actual - expected))
    assert actual == expected, label
    print("PASS:", label)


## 4. Test basic working-memory matching

**Action:** ask the original WM for `parent(mary, ?child)`.

**Expected:** Ann and Bob, and nobody else. Asking is matching existing facts; it does not apply production rules.

**Logic:** `?child` receives different bindings for different matches. A set has no guaranteed iteration order, so sort for presentation and compare answer sets.


In [5]:
wm_initial = WorkingMemory(TEST_1_FACTS)

In [6]:

answers = wm_initial.ask(("parent", "mary", "?child"))

In [7]:
for b in answers:
    print(b)

{'?child': 'bob'}
{'?child': 'ann'}


In [8]:

children = {substitute(("parent", "mary", "?child"), b)
            for b in wm_initial.ask(("parent", "mary", "?child"))}
explain_check("Mary's children", children,
              {("parent", "mary", "ann"), ("parent", "mary", "bob")})
show_facts(children)
assert not wm_initial.holds(("grandparent", "mary", "sam"))
print("Grandparent relationship is not an initial fact; it must be inferred.")


PASS: Mary's children
('parent', 'mary', 'ann')
('parent', 'mary', 'bob')
Grandparent relationship is not an initial fact; it must be inferred.


## 5. Test Forward chaining

**Action:** create fresh WM, apply the three rules repeatedly, and print the new facts.

**Expected:** 13 new facts and 19 facts in total, exactly matching `EXPECTED_CLOSURE`.

**Logic:**

Illustrative dependency sequence:

- Original parent facts support the three grandparent facts and six direct ancestor facts.
- Those ancestor facts support `ancestor(mary, sam)`, `ancestor(mary, ivy)` and `ancestor(ann, leo)`.
- The recursive rule also establishes `ancestor(mary, leo)`.
- Eventually a full pass adds nothing: the fixpoint.

The engine has to join all rule conditions under consistent bindings, substitute those bindings into the conclusion, and keep going until no new ground fact appears. 


In [9]:
wm_forward = WorkingMemory(TEST_1_FACTS)
before_forward = wm_forward.snapshot()
forward_chain(wm_forward, TEST_1_RULES)
new_facts = wm_forward.facts - before_forward
print("Derived facts:")
show_facts(new_facts)
print("Initial:", len(before_forward), "New:", len(new_facts),
      "Total:", len(wm_forward.facts))
explain_check("Exact forward closure", wm_forward.snapshot(), EXPECTED_CLOSURE)
assert before_forward <= wm_forward.facts
print("PASS: original facts are preserved (monotonic growth).")


Derived facts:
('ancestor', 'ann', 'leo')
('ancestor', 'ann', 'sam')
('ancestor', 'bob', 'ivy')
('ancestor', 'mary', 'ann')
('ancestor', 'mary', 'bob')
('ancestor', 'mary', 'ivy')
('ancestor', 'mary', 'leo')
('ancestor', 'mary', 'sam')
('ancestor', 'sam', 'leo')
('ancestor', 'tom', 'nina')
('grandparent', 'ann', 'leo')
('grandparent', 'mary', 'ivy')
('grandparent', 'mary', 'sam')
Initial: 6 New: 13 Total: 19
PASS: Exact forward closure
PASS: original facts are preserved (monotonic growth).


## 6. Test Forward chaining Termination

**Action:** run forward chaining again on the completed WM.

**Expected:** the second run finishes and adds nothing. Re-deriving an existing fact must not count as progress.

**Logic:** with this finite domain, ground facts and no generation of new constants, there are finitely many possible facts. The set prevents duplicates, and the loop stops when nothing new is added.


In [10]:
closure_before_rerun = wm_forward.snapshot()
forward_chain(wm_forward, TEST_1_RULES)
assert wm_forward.snapshot() == closure_before_rerun
print("PASS: second run adds no facts; fixpoint remains at 19 facts.")


PASS: second run adds no facts; fixpoint remains at 19 facts.


## 7. Test Backward chaining

**Action:** on a NEW WM containing only the six original parent facts, query `grandparent(mary, sam)`.

**Expected:** success, even though no grandparent fact is stored initially.

**Logic:**:

1. Match the query against the grandparent rule's conclusion.
2. Establish `parent(mary, ?middle)` and `parent(?middle, sam)`.
3. The first condition has two candidates: Ann and Bob.
4. Ann also satisfies the second condition; Bob does not.
5. At least one full proof means the goal succeeds.


In [11]:
wm_backward = WorkingMemory(TEST_1_FACTS)
backward_baseline = wm_backward.snapshot()
query = ("parent", "mary", "ann")
answers = backward_answers(wm_backward, query, TEST_1_RULES)
explain_check("True ground goal", answers, {query})
assert wm_backward.snapshot() == backward_baseline
print("PASS: backward proof leaves WM unchanged.")


PASS: True ground goal
PASS: backward proof leaves WM unchanged.


In [12]:
wm_backward = WorkingMemory(TEST_1_FACTS)
backward_baseline = wm_backward.snapshot()
query = ("grandparent", "mary", "sam")
answers = backward_answers(wm_backward, query, TEST_1_RULES)
explain_check("True ground goal", answers, {query})
assert wm_backward.snapshot() == backward_baseline
print("PASS: backward proof leaves WM unchanged.")


PASS: True ground goal
PASS: backward proof leaves WM unchanged.


## 8. Test Backward chaining Negative (Unsupported Goal)

**Action:** query `grandparent(mary, leo)`.

**Expected:** no proof, because Leo is three parent links from Mary. The same pair WILL satisfy ancestry, so this tests that the rule predicates are respected.

“No proof” means the query is not established by this rule base and these facts. The engine has not derived an explicit negative fact.


In [13]:
query = ("grandparent", "mary", "leo")
explain_check("Unsupported grandparent goal", backward_answers(wm_backward, query, TEST_1_RULES), set())
print("The query is not established by the current facts and rules.")


PASS: Unsupported grandparent goal
The query is not established by the current facts and rules.


## 9. Test Backward chaining multiple answers

**Action:** query `grandparent(mary, ?who)`.

**Expected:** exactly Ivy and Sam. Returning only the first proof is insufficient for a query with multiple answers.

**Logic:** alternatives must be explored without allowing bindings from one failed branch to contaminate another.


In [14]:
query = ("grandparent", "mary", "?who")
answers = backward_answers(wm_backward, query, TEST_1_RULES)
explain_check("All grandchildren", answers,
              {("grandparent", "mary", "ivy"), ("grandparent", "mary", "sam")})
show_facts(answers)


PASS: All grandchildren
('grandparent', 'mary', 'ivy')
('grandparent', 'mary', 'sam')


## 10. Backward chaining: a recursive proof

**Action:** query `ancestor(mary, leo)` on the original facts.

**Expected:** success through Mary → Ann → Sam → Leo.

**Logic:**:

1. To prove `ancestor(mary, leo)`, use a parent link to Ann and prove `ancestor(ann, leo)`.
2. Use Ann's parent link to Sam and prove `ancestor(sam, leo)`.
3. Use the direct ancestry rule and the stored fact `parent(sam, leo)`.

Each application of the recursive rule needs fresh variable names. Its `?x`, `?y` and `?z` must not accidentally share bindings with another application of the same rule. Freshening changes internal names, not the rule's meaning.


In [15]:
query = ("ancestor", "mary", "leo")
explain_check("Recursive ancestor proof", backward_answers(wm_backward, query, TEST_1_RULES), {query})


PASS: Recursive ancestor proof


## 11. Test recursive variable queries and disconnected families

**Action:** ask for all Mary's descendants, then all Leo's ancestors, then an unsupported relationship to Nina.

**Expected:**

| Query | Answers |
|---|---|
| `ancestor(mary, ?who)` | ann, bob, ivy, leo, sam |
| `ancestor(?who, leo)` | ann, mary, sam |
| `ancestor(mary, nina)` | no proof |
| `ancestor(mary, mary)` | no proof in this acyclic fixture |

These ancestry rules describe paths of at least one parent link, not automatic self-ancestry.


In [16]:
query = ("ancestor", "mary", "?who")
expected = {("ancestor", "mary", name) for name in ("ann", "bob", "ivy", "leo", "sam")}
answers = backward_answers(wm_backward, query, TEST_1_RULES)
explain_check("All Mary's descendants", answers, expected)
show_facts(answers)

query = ("ancestor", "?who", "leo")
expected = {("ancestor", name, "leo") for name in ("ann", "mary", "sam")}
explain_check("All Leo's ancestors", backward_answers(wm_backward, query, TEST_1_RULES), expected)

for query in [("ancestor", "mary", "nina"), ("ancestor", "mary", "mary")]:
    explain_check(str(query), backward_answers(wm_backward, query, TEST_1_RULES), set())
assert wm_backward.snapshot() == backward_baseline
print("PASS: all backward queries leave WM unchanged.")


PASS: All Mary's descendants
('ancestor', 'mary', 'ann')
('ancestor', 'mary', 'bob')
('ancestor', 'mary', 'ivy')
('ancestor', 'mary', 'leo')
('ancestor', 'mary', 'sam')
PASS: All Leo's ancestors
PASS: ('ancestor', 'mary', 'nina')
PASS: ('ancestor', 'mary', 'mary')
PASS: all backward queries leave WM unchanged.


## 12. Compare the two reasoning directions

**Action:** ask both reasoners for all ancestor pairs and all grandparent pairs.

**Expected:** identical answers for each predicate: ten ancestor pairs and three grandparent pairs.

Forward chaining starts with facts and derives conclusions. Backward chaining starts with a goal and seeks supporting facts. They use the SAME `DEMO_RULES`, and should agree on this bounded demonstration domain.

Agreement alone is not enough: the earlier checks against independently listed expected facts protect against both implementations making the same mistake.


In [17]:
for predicate in ("ancestor", "grandparent"):
    query = (predicate, "?a", "?b")
    forward_answers = {substitute(query, b) for b in wm_forward.ask(query)}
    backward_result = backward_answers(wm_backward, query, TEST_1_RULES)
    explain_check(f"Forward/backward agreement: {predicate}",
                  backward_result, forward_answers)
    print("Answer count:", len(backward_result))


PASS: Forward/backward agreement: ancestor
Answer count: 10
PASS: Forward/backward agreement: grandparent
Answer count: 3


## 13. Test engine/knowledge separation

**Action:** add a new rule AS DATA, with no edit to the engine:

`IF grandparent(?a, ?b) THEN two_generations_above(?a, ?b)`

**Expected:** both reasoners support the new predicate; Mary is two generations above Sam. This is a simple demonstration that the generic engine is not hardcoded to the original predicates.

**Logic:**
Use a separate rule list and fresh memories so this experiment does not alter the earlier fixtures.


In [18]:
extended_rules = TEST_1_RULES + [
    {"if": [("grandparent", "?a", "?b")],
     "then": ("two_generations_above", "?a", "?b")}
]
wm_extended_forward = WorkingMemory(TEST_1_FACTS)
forward_chain(wm_extended_forward, extended_rules)
expected_new = {("two_generations_above", fact[1], fact[2])
                for fact in EXPECTED_GRANDPARENTS}
actual_new = {f for f in wm_extended_forward.facts if f[0] == "two_generations_above"}
explain_check("New rule works without engine edits", actual_new, expected_new)

wm_extended_backward = WorkingMemory(TEST_1_FACTS)
query = ("two_generations_above", "mary", "sam")
answers = {substitute(query, b)
           for b in backward_chain(wm_extended_backward, extended_rules, query)}
explain_check("Backward chaining supports new predicate", answers, {query})


PASS: New rule works without engine edits
PASS: Backward chaining supports new predicate
